# NLU Shared Task - Demo Solution 2 (Category C)

This notebook demonstrates how to generate final predictions using the Solution 2 Neural Meta-Learner Ensemble. It expects that the individual transformer cross-encoders have already produced their probability CSVs.

## Overview: Training vs. Inference

**This notebook contains DEMO/INFERENCE CODE ONLY.** It runs the trained ensemble model to generate predictions from test data.

- **Training code** (model building) is located in `src/solution2/` with subdirectories for each component:
  - `deberta_model/` - DeBERTa transformer model training
  - `electra_model/` - ELECTRA transformer model training
  - `roberta_model/` - RoBERTa transformer model training
  - `xlnet_model/` - XLNet transformer model training
  - `ensemble/` - Meta-learner ensemble training

- **This notebook** (inference code) loads the pre-trained models and generates predictions on test data.

In [ ]:
!pip install pandas gdown torch transformers scikit-learn numpy

In [ ]:
import pandas as pd
from pathlib import Path
import sys
import os

# Ensure src is in the python path
sys.path.append(os.path.abspath('..'))

## Prerequisites for Running This Notebook

To run this demo notebook, you need:

1. **Pre-trained model files**: Download the trained models from Google Drive (see next cell)
2. **Base model probability files**: Download the base transformer model predictions (see next cell)
3. **Installed dependencies**: Run the pip install cell above

The notebook will download these automatically, but alternatively you can:
- Place trained models in `models/solution2/`
- Place model predictions in `outputs/solution2/`

Once set up, this notebook will:
1. Load the pre-trained ensemble meta-learner
2. Load probability predictions from the 4 base transformer models (DeBERTa, ELECTRA, RoBERTa, XLNet)
3. Pass them through the meta-learner for final ensemble predictions
4. Apply calibration and output the final predictions CSV

## Step 1: Download Pre-trained Models and Base Predictions
Before running the ensemble inference, we need to download the base model predictions from Google Drive and set up the outputs directory. This only needs to run once.

**What gets downloaded:**
- Base model probability files: Predictions from the 4 individual transformer models (DeBERTa, ELECTRA, RoBERTa, XLNet) on train/dev/test splits
- Trained models: The ensemble meta-learner weights and calibrators

In [ ]:
# Only run this cell if the data and models have not been downloaded yet
import gdown

output_dir = Path('../outputs/solution2')
output_dir.mkdir(parents=True, exist_ok=True)
print(f'Created output directory: {output_dir}')

# Download model predictions from Google Drive
gdrive_folder_url = "https://drive.google.com/drive/folders/1saMnwl_u3_FZMiDiWzap-eyhmXymznFx?usp=sharing"
print(f'Downloading model predictions from Google Drive...')

gdown.download_folder(url=gdrive_folder_url, output=str(output_dir), quiet=False, use_cookies=False)
print(f'Downloaded model predictions to {output_dir}')

models_dir = Path('../models/solution2')
models_dir.mkdir(parents=True, exist_ok=True)
print(f'Created models directory: {models_dir}')

# Download trained ensemble models from Google Drive
gdrive_models_url = "https://drive.google.com/drive/folders/1arKIOSEAZxAz4P_-MotKRo0aFjFg-QGp?usp=sharing"
print(f'Downloading trained ensemble models from Google Drive...')

gdown.download_folder(url=gdrive_models_url, output=str(models_dir), quiet=False, use_cookies=False)
print(f'Downloaded trained models to {models_dir}')


In [ ]:
## Setup: Configure Paths and Verify Files

# Define required paths
OUTPUT_DIR = Path('../outputs/solution2')
MODELS_DIR = Path('../models/solution2')

# Create directories if they don't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Base models to check
BASE_MODELS = ['deberta_model', 'electra_model', 'roberta_model', 'xlnet_model']

# Sanity checks
print('Checking required files...\n')

# Check that base model probability files exist for all necessary splits
for model_name in BASE_MODELS:
    model_probs_dir = OUTPUT_DIR / model_name
    assert model_probs_dir.exists(), f'{model_name} probabilities directory not found: {model_probs_dir}'
    
    for split in ['train', 'dev', 'test']:
        probs_file = model_probs_dir / f'probs_{split}.csv'
        assert probs_file.exists(), f'{model_name} probs_{split}.csv not found: {probs_file}'
    
    print(f'{model_name}: all splits (train/dev/test) found')

# Check ensemble directory
ensemble_dir = OUTPUT_DIR / 'ensemble'
assert ensemble_dir.exists(), f'Ensemble directory not found: {ensemble_dir}'
assert (ensemble_dir / 'probs_dev.csv').exists(), f'Ensemble probs_dev.csv not found'
print(f'ensemble: base probabilities found')

# Check trained ensemble models
assert (MODELS_DIR / 'ensemble' / 'meta_learner.pt').exists(), f'meta_learner.pt not found in {MODELS_DIR / "ensemble"}'
calibrators_dir = MODELS_DIR / 'ensemble' / 'calibrators'
assert calibrators_dir.exists(), f'Calibrators directory not found: {calibrators_dir}'
print(f'ensemble models: meta_learner and calibrators found')

print(f'\nAll required files present. Ready to run ensemble inference.')

## 1. Run Ensemble Inference
We use the predefined ensemble prediction script. This will load the base probabilities for the specified split (e.g. `dev` or `test`), pass them through the 4-layer fully connected meta-learner, and calibrate the outputs.

In [ ]:
from src.solution2.ensemble.predict_ensemble import predict_probs

split_name = 'test' # Change to 'dev' to evaluate on development set, or 'test' for final test set

print(f'Running Neural Meta-Ensemble on split: {split_name}...')
print(f'This loads base probability predictions from the 4 transformer models')
print(f'and passes them through the trained meta-learner ensemble...')
results_df = predict_probs(split=split_name)

print(f'\nGenerated {len(results_df)} predictions.')

## 2. Format and Save Submission
We extract only the `prediction` column (0 or 1) as required by the shared task spec and save it.

In [ ]:
submission_df = results_df[['prediction']].copy()

# Preview output
display(submission_df.head())

# Save to disk
out_path = OUTPUT_DIR / f'predictions_demo_{split_name}.csv'
submission_df.to_csv(out_path, index=False)
print(f'Saved predictions to {out_path}')

## Where to Find the Training Code

For the marking team: If you want to see how the individual base models and ensemble were trained, refer to:

- **DeBERTa model training**: `src/solution2/deberta_model/` - see `model.py` and training scripts
- **ELECTRA model training**: `src/solution2/electra_model/` - see `model.py` and training scripts  
- **RoBERTa model training**: `src/solution2/roberta_model/` - see `model.py` and training scripts
- **XLNet model training**: `src/solution2/xlnet_model/` - see `model.py` and training scripts
- **Ensemble meta-learner training**: `src/solution2/ensemble/` - see training code for the meta-learner model

The README files in each subdirectory explain the training procedure and key hyperparameters.